In [1]:
import os
import torch
import math
import torch.nn as nn
import sentencepiece as spm
from dataclasses import dataclass


In [2]:
print(torch.__version__)

2.14.0+cu130


In [3]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

In [4]:
# ====================================================
# CFG
# ====================================================
@dataclass
class CFG:
    d_model = 512
    h=8
    d_ff=2048
    dropout=0.1
    tokenizer_src_loc = './sentencepiece_tok/en_hi_shared.model'
    tokenizer_tgt_loc = './sentencepiece_tok/en_hi_shared.model'
    src_seq_len = 100
    tgt_seq_len = 120
    src_lang = 'en'
    tgt_lang = 'hi'
    model_dir = 'models'
    prepared_dataset_dir = './tokenized_dataset/'
    prepared_valid_dataset_name = 'valid_dataset.pt'
    best_state = 'best_state_9.pth'
    max_token_gen_len = 100 #total tokens to generate for validation

    def __post_init__(self):
        # decode() indexes pe at positions 0..max_token_gen_len-1, so the
        # positional embedding table (tgt_seq_len rows) must cover them.
        assert self.max_token_gen_len <= self.tgt_seq_len, (
            f'max_token_gen_len ({self.max_token_gen_len}) must be <= '
            f'tgt_seq_len ({self.tgt_seq_len})'
        )


In [5]:
class InputEmbeddings(nn.Module):
    def __init__(self, embed_dim: int, vocab_size: int):
        "In paper embed_dim is d_model, which is 512"
        super().__init__()
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
    def forward(self,x):
        return self.embeddings(x)*math.sqrt(self.embed_dim)

In [6]:
class PositionalEmbeddings(nn.Module):
    def __init__(self, embed_dim: int, seq_len: int, dropout: float):
        super().__init__()        
        self.embed_dim = embed_dim
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        pos = torch.arange(0, seq_len).unsqueeze(1)
        #log exp for numerical stability
        div = torch.exp(-2*(torch.arange(0, embed_dim))/embed_dim*math.log(10000))
        pe = pos*div
        pe[:, 0::2] = torch.sin(pe[:,0::2])
        pe[:, 1::2] = torch.cos(pe[:,1::2])
        pe = pe.unsqueeze(0)
        self.register_buffer('pe',pe)
        
    def forward(self,x, pos=None):
        if pos is None:
            x = x+(self.pe[:, torch.arange(0, x.shape[1]),:]).requires_grad_(False)
        else:
            x = x+(self.pe[:, pos,:]).requires_grad_(False)

        return self.dropout(x)

In [7]:
class FFN(nn.Module):
    
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
    def forward(self,x):
        return self.feed_forward(x)

In [8]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model: int, h: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.h = h
        
        assert d_model%h ==0, "d_model should be divisible by h"
        
        self.d_k = d_model//h
        self.w_q = nn.Linear(d_model,d_model, bias= False)
        self.w_k = nn.Linear(d_model,d_model, bias= False)        
        self.w_v = nn.Linear(d_model, d_model, bias= False)  
        
        self.w_o = nn.Linear(d_model, d_model, bias= False)  
        self.dropout = nn.Dropout(dropout)
        
    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        scores = query@key.transpose(-2,-1)/math.sqrt(d_k)
        if mask is not None:
            scores.masked_fill_(mask==0, -1e9)
        scores = scores.softmax(dim=-1)
        
        ### do we need ?
        if dropout is not None:
            scores = dropout(scores)
            
        return scores@value
        
    def forward(self, q, k, v, mask, prev_kv = None, is_cross_attention=False):
        # batch, seq len, embed dim
        query = self.w_q(q)
        #for cross attn k,v comes from encoder and computed once
        if is_cross_attention and prev_kv is not None:
            key, value = prev_kv
        else:
            key = self.w_k(k)
            value = self.w_v(v)
            if prev_kv is not None:
                pastk, pastv = prev_kv
                key = torch.cat([pastk, key], dim=1)
                value = torch.cat([pastv, value], dim=1)

        present_kv  = (key, value)
            
        # batch, h, seq, d_k
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1,2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1,2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1,2)
        
        x = self.attention(query, key, value, mask, self.dropout)
        
        x = x.transpose(1,2).contiguous().view(x.shape[0], -1, self.h*self.d_k)
        
        return self.w_o(x), present_kv
        

In [10]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model:int, h:int, d_ff:int, dropout:float):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, h, dropout)
        self.ffn = FFN(d_model, d_ff, dropout)
        self.layer_norm_1 = nn.LayerNorm(d_model)
        self.layer_norm_2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        x_ln = self.layer_norm_1(x)
        attention, kv = self.attention(x_ln, x_ln, x_ln, mask, None)
        x = x + self.dropout(attention)
        ffn_out = self.ffn(self.layer_norm_2(x))
        x = x+ self.dropout(ffn_out)
        
        return x

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model:int, h:int, d_ff:int, dropout:float):
        super().__init__()
        self.attention_1 = MultiHeadAttention(d_model, h, dropout)
        self.attention_2 = MultiHeadAttention(d_model, h, dropout)
        self.ffn = FFN(d_model, d_ff, dropout)
        self.layer_norm_1 = nn.LayerNorm(d_model)
        self.layer_norm_2 = nn.LayerNorm(d_model)
        self.layer_norm_3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.kv_1 = None #self attn
        self.kv_2 = None #cross attn
        
    def forward(self, x, encoder_out, src_mask, tgt_mask):
        x_ln = self.layer_norm_1(x)
        attention, self.kv_1 = self.attention_1(x_ln, x_ln, x_ln, tgt_mask, self.kv_1)
        x = self.layer_norm_2(x + self.dropout(attention))

        attention, self.kv_2 = self.attention_2(x, encoder_out, encoder_out,\
                                                 src_mask, self.kv_2, True)
        x = x + self.dropout(attention)
        
        ffn_out = self.ffn(self.layer_norm_3(x))
        x = x+ self.dropout(ffn_out)
        
        return x

    def reset(self):
        self.kv_1 = None
        self.kv_2 = None
        

In [12]:
class ProjectionLayer(nn.Module):
    def __init__(self, d_model:int, vocab_size: int):
        super().__init__()
        
        self.proj = nn.Linear(d_model, vocab_size)
        
    def forward(self,x):
        #for numerical stability!!!
        return self.proj(x)


In [27]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size: int, tgt_vocab_size:int, src_seq_len:int, tgt_seq_len:int, d_model:int, h:int, d_ff:int, dropout:float, N=6):
        super().__init__()
        
        self.encoder = nn.ModuleList(
            [
                EncoderBlock(d_model, h, d_ff, dropout)
                for _ in range(N)
            ]
        )

        self.decoder = nn.ModuleList(
            [
                DecoderBlock(d_model, h, d_ff, dropout)
                for _ in range(N)
            ]
        )
        
        self.src_embed = InputEmbeddings(d_model, src_vocab_size)
        self.tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)
        self.src_pos_embed = PositionalEmbeddings(d_model, src_seq_len, dropout)
        self.tgt_pos_embed = PositionalEmbeddings(d_model, tgt_seq_len, dropout)
        self.projection_layer = ProjectionLayer(d_model, tgt_vocab_size)
        
    
    def encode(self,x, src_mask):
        x = self.src_embed(x)
        x = self.src_pos_embed(x)
        for layer in  self.encoder:
            x = layer(x, src_mask)
        return x
        
    def decode(self, x, encoder_out,src_mask, tgt_mask, pos):
        x = x[:,-1].unsqueeze(1)
        x = self.tgt_embed(x)
        x = self.tgt_pos_embed(x, pos)
        for layer in  self.decoder:
            x = layer(x, encoder_out,src_mask,tgt_mask)
        return x
        
    def reset_kv(self):
        for block in self.decoder:
            block.reset()        
    

In [28]:
config = CFG()

In [29]:
tokenizer_src = spm.SentencePieceProcessor()
tokenizer_src.load(config.tokenizer_src_loc)

True

In [30]:
model = Transformer(
    src_vocab_size= tokenizer_src.vocab_size(),
    tgt_vocab_size= tokenizer_src.vocab_size(),
    src_seq_len=config.src_seq_len,
    tgt_seq_len=config.tgt_seq_len,
    d_model=config.d_model,
    h=config.h,
    d_ff=config.d_ff,
    dropout=config.dropout
    )

In [32]:
state = torch.load(os.path.join(config.model_dir, config.best_state), map_location=device)
# checkpoint was saved from a torch.compile'd model, so its keys carry an
# '_orig_mod.' prefix; strip it to load into the plain module
state_dict = {k.replace('_orig_mod.', '', 1): v for k, v in state['model_state_dict'].items()}
model.load_state_dict(state_dict)


<All keys matched successfully>

In [33]:
model.eval()
model.to(device)
print('')

In [34]:
class DatasetInference():
    def __init__(self, tokenizer_src, tokenizer_tgt,src_seq_len, tgt_seq_len):
        self.tokenizer_src = tokenizer_src
        self.tokenizer_tgt = tokenizer_tgt
        self.src_seq_len = src_seq_len
        self.tgt_seq_len = tgt_seq_len
        self.sos_token = torch.tensor([tokenizer_src.bos_id()], dtype = torch.int32)
        self.eos_token = torch.tensor([tokenizer_src.eos_id()], dtype = torch.int32)
        self.pad_token = torch.tensor([tokenizer_src.pad_id()], dtype = torch.int32)
        

    def prepare_data(self, txt):
        txt_encoded = torch.tensor(self.tokenizer_src.encode(txt), dtype=torch.int32)
        encoder_tokens = torch.ones(self.src_seq_len, dtype=torch.int32)*self.pad_token
        encoder_tokens[0]= self.sos_token
        encoder_tokens[1:len(txt_encoded)+1] = txt_encoded
        encoder_tokens[len(txt_encoded)+1] = self.eos_token
        
        encoder_mask = (encoder_tokens!=self.pad_token).unsqueeze(0).unsqueeze(0).bool()
        return encoder_tokens.unsqueeze(0), encoder_mask


In [43]:
def translate(config, txt, tokenizer_src, tokenizer_tgt, model, dataset, device):
    encoder_tokens, encoder_mask = dataset.prepare_data(txt)
    eos_token = tokenizer_tgt.eos_id()
    with torch.no_grad():
        decoder_inp = torch.empty(1,1).fill_(tokenizer_tgt.bos_id()).type(torch.int32).to(device)
        encoder_inp = encoder_tokens.to(device)
        encoder_mask = encoder_mask.to(device)
        model.reset_kv()
        encoder_out = model.encode(encoder_inp, encoder_mask)

        for pos in range(config.max_token_gen_len):
            # with a kv cache the single query token attends to every cached
            # key, which is already causal -> no target mask needed
            decoder_mask = None
            decoder_out = model.decode(decoder_inp, encoder_out, encoder_mask, decoder_mask,pos)
            logits = model.projection_layer(decoder_out)
            next_idx = logits.argmax(dim=-1).item()
            next_idx_tensor = torch.empty(1,1).fill_(next_idx).type(torch.int32).to(device)
            decoder_inp = torch.cat((decoder_inp, next_idx_tensor), dim=1)
            if next_idx == eos_token:
                break

    pred_txt = tokenizer_tgt.decode(decoder_inp.squeeze().cpu().tolist())
    return pred_txt, decoder_inp.squeeze().cpu().numpy()            

In [44]:
dataset = DatasetInference(tokenizer_src, tokenizer_src, config.src_seq_len, config.tgt_seq_len)

In [45]:
txt = "Hello! How are you? I am Neeraj, I am testing my translation model"
pred_txt, _ = translate(config, txt, tokenizer_src, tokenizer_src, model, dataset, device)
print(pred_txt)

होलो! तुम कैसे हो? मैं नीरज हूँ, मैं अपने अनुवाद मॉडल की जांच कर रहा हूँ
